In [1]:
# ======================================================
# thesis 解析ノートブック共通の設定 (各ノートブックの先頭セルの run_nb で読み込む)
# モデルや範囲を変えるときはここだけ変更し, ノートブックの先頭セルから実行し直す
# ======================================================
import glob
import linecache
import math
import os
import re
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.ticker as ptick
from matplotlib.colors import LogNorm, SymLogNorm
from matplotlib.lines import Line2D
from scipy import interpolate
from scipy.ndimage import gaussian_filter, gaussian_filter1d
from scipy.signal import find_peaks
from scipy.special import i0e

In [ ]:
# ===== 解析設定 =====
MODEL = "/test1e17"
BASE_DIR = "/mnt/hdd/shimokawa"
r_min, r_max = 0.55, 300.0     # 動径方向の表示範囲 [au]
t_min, t_max = 0.0, 100.0     # 時間方向の表示範囲 [10^3 yr]
r_search_max = 200.0          # ギャップ外縁を探す範囲の上限 [au]
t_gap_min = 10.0              # ギャップの解析 (冪指数など) に使う最初の時刻 [10^3 yr]. これより前の時刻は使わない
T_snow = 160.0                # スノーラインの温度 [K]

# ディレクトリ名 -> 図に表示するモデル名
model_names = {
    "/test1e18": "CR18",
    "/test1e17": "CR17",
    "/test1e16": "CR16",
    "/test1e15": "CR15",
}
# model_names = {
#     "/s5beta4_zeta1e17": r"$ \zeta_{\rm CR,0}=1.0 \times 10^{-17}\,{\rm s^{-1}}$",
#     "/s5beta4_zeta1e16": r"$ \zeta_{\rm CR,0}=1.0 \times 10^{-16}\,{\rm s^{-1}}$",
#     "/s5beta4_zeta1e15": r"$\zeta_{\rm CR,0}=1.0 \times 10^{-15}\,{\rm s^{-1}}$",
# }
# ディレクトリ名 -> 図の保存先 (./fig 以下)
FIG_TAGS = {
    "/test1e18": "b4z18",
    "/test1e17": "b4z17",
    "/test1e16": "b4z16",
    "/test1e15": "b4z15",
}
FIG_DIR = "./fig/" + FIG_TAGS[MODEL]
os.makedirs(FIG_DIR, exist_ok=True)

# ===== 図の保存先 =====
# 1 つのモデルの図は <root>/Result, 複数のモデルを比べる図は <root>/Discussion に保存する.
# root : 論文用 (phd.ipynb) は PHD_FIG_DIR, 解析用 (analysis_*.ipynb) は ANALYSIS_FIG_DIR
# ファイル名の先頭にはモデルのタグ (FIG_TAGS) を自動で付ける (例: Result/b4z16_sigT.png , Discussion/b4z16_b4z15_alpha.png)
PHD_FIG_DIR = "./phd_fig"         # 論文用の図
ANALYSIS_FIG_DIR = "./fig"        # 解析用の図

def fig_path(name, models=None, root=None):
    # 図の保存先のパスを返す (フォルダがなければ作る)
    #   name   : ファイル名 (例: "sigT.png")
    #   models : モデルのリスト. ディレクトリ名 ("/test1e16") か RUNS の要素 (dict) を入れる. None なら MODEL 1 つ
    #   root   : 保存先の親フォルダ. None なら PHD_FIG_DIR
    if models is None:
        models = [MODEL]
    keys = ["/" + m["key"] if isinstance(m, dict) else m for m in models]
    tags = [FIG_TAGS.get(k, k.strip("/")) for k in keys]
    sub = "Result" if len(tags) == 1 else "Discussion"
    out_dir = os.path.join(PHD_FIG_DIR if root is None else root, sub)
    os.makedirs(out_dir, exist_ok=True)
    return os.path.join(out_dir, "_".join(tags) + "_" + name)

# ===== ギャップの判定条件 (gap_detection.ipynb と compare_beta_ini.ipynb で共通) =====
Lambda_obs = 1300.0        # 観測波長 [micron] (Band 6)
flux_frac = 0.95           # r_disk の定義 (全フラックスに対する割合)
use_delta_cut = True       # True: delta_I >= delta_min を課す
delta_min = 1.5
use_sigma_cut = False      # True: Sigma_g/Sigma_g,ini < sigma_ratio_max を課す
sigma_ratio_max = 0.5
use_rdisk_cut = True       # True: r_ring <= r_disk を課す
use_width_cut = True       # True: r_ring - r_gap >= w_min_H * H_g(r_gap) を課す
w_min_H = 2.0
use_beam = False           # True: ビーム畳み込み後の I_nu で判定
beam_fwhm_arcsec = 0.05    # ビーム FWHM [arcsec]
distance_pc = 140.0        # 距離 [pc]
dr_conv = 0.2              # 畳み込み用の等間隔格子 [au]
r_conv_max = 500.0         # 畳み込み用格子の外端 [au]

# ===== Sigma のリング・ギャップの判定条件 (analysis_obs_theory.ipynb と analysis_snapshot.ipynb で共通) =====
r_det_min = 2.0            # 判定の内端 [au]
sig_delta_min = 1.1        # 深さ Sigma(r_ring)/Sigma(r_gap) の下限
sig_w_min_H = 1.0          # r_ring - r_gap >= sig_w_min_H * H_g(r_gap)
D_SIGMA = 2.0              # D = d ln Sigma/dr (prominence 判定では ln Sigma) にかけるガウシアンフィルター幅 [セル数]
use_prominence = True      # True: find_peaks の prominence で極値を選ぶ, False: D の符号変化
prom_min = 0.1             # ln Sigma の prominence の下限 (Sigma で e^prom_min 倍の落ち込み)

# ===== 図の文字サイズ (plt.rcParams.update(PLOT_STYLE) で適用) =====
PLOT_STYLE = {
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 12,
    "figure.titlesize": 16,
}

# ===== 複数パネルの図に (a), (b), (c), ... を付ける =====
# 各パネルの枠の内側の左上に付ける. 表示と savefig の両方で自動的に付く. 付けない図は fig._no_panel_labels = True
import matplotlib.figure
PANEL_LABEL_STYLE = dict(fontsize=18, fontweight="bold", ha="left", va="top",
                         bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7))

def label_panels(fig):
    # カラーバー, 非表示の軸, twinx で重ねた軸 (同じ subplotspec) は数えない
    panels, seen = [], set()
    for ax in fig.axes:
        if not ax.get_visible() or not ax.axison or hasattr(ax, "_colorbar"):
            continue
        ss = ax.get_subplotspec()
        if ss is not None:
            ss = ss.get_topmost_subplotspec()  # colorbar で入れ子になった gridspec でも外側の位置を使う
            key = (ss.rowspan.start, ss.colspan.start)
        else:
            pos = ax.get_position()
            key = (-round(pos.y1, 2), round(pos.x0, 2))
        if key in seen:
            continue
        seen.add(key)
        panels.append((key, ax))
    if len(panels) < 2:
        return
    # 行優先: 上の段から, 同じ段では左から
    panels.sort(key=lambda p: p[0])
    for i, (_, ax) in enumerate(panels):
        if getattr(ax, "_panel_label", None) is None:
            ax._panel_label = ax.text(0.02, 0.98, f"({chr(ord('a') + i)})", transform=ax.transAxes,
                                      zorder=10, **PANEL_LABEL_STYLE)

if not hasattr(matplotlib.figure.Figure, "_draw_orig"):
    matplotlib.figure.Figure._draw_orig = matplotlib.figure.Figure.draw
    def _draw_with_panel_labels(self, renderer):
        if not getattr(self, "_no_panel_labels", False):
            label_panels(self)
        return matplotlib.figure.Figure._draw_orig(self, renderer)
    matplotlib.figure.Figure.draw = _draw_with_panel_labels

In [2]:
# ===== 物理定数 (cgs). セル内で別の値を定義している場合はセルの値が優先される =====
Msolar = 1.989000e+33
G = 6.67408e-08
AU = 1.495979e+13
kB = 1.38064852e-16
mH = 1.67353e-24
mu = 2.34
yr = 3.155760e+07
e = 4.803204673e-10
h_bar = 6.6260755e-27
sigma_SB = 5.6705e-5
light_speed = 2.99792458e10


def list_years(path_gas, t_start=0, path_mag=None):
    # path_gas 内の out<year>_gas の year を昇順で返す. path_mag を渡すと out<year>_B もある時刻だけにする
    return sorted(int(m.group(1)) for f in os.listdir(path_gas)
                  if (m := re.fullmatch(r"out(\d+)_gas", f)) and int(m.group(1)) >= t_start
                  and (path_mag is None or os.path.exists(path_mag + "/out{}_B".format(m.group(1)))))


def find_snowline(r, T):
    # スノーライン [au]: T が T_snow を横切る最も外側の半径 (log r で線形補間). なければ NaN
    k = np.where((T[:-1] - T_snow) * (T[1:] - T_snow) <= 0.0)[0]
    if len(k) == 0:
        return np.nan
    k = k[-1]
    x = (T_snow - T[k]) / (T[k + 1] - T[k])
    return 10.0**(np.log10(r[k]) + x * (np.log10(r[k + 1]) - np.log10(r[k]))) / AU

def read_input(path):
    # 計算時の input から M_star [g], eta_max を読む
    M_star, eta_max = Msolar, 1.0e20
    with open(path + "/input") as f:
        for line in f:
            key = line.split("//")[0].split("=")[0].strip()
            if key == "star_mass":
                M_star = float(line.split("//")[0].split("=")[1]) * Msolar
            elif key == "eta_max":
                eta_max = float(line.split("//")[0].split("=")[1])
    return M_star, eta_max


# ===== ギャップの判定: I_dust (定義は gap_detection.ipynb) =====
def calc_Kabs(dradius):
    # 各ダストビンの吸収係数 (optool -dsharp, 波長 Lambda_obs)
    os.environ["PATH"] = os.path.expanduser("~/bin") + ":" + os.environ["PATH"]
    Kabs = []
    for a in dradius:
        cmd = "optool -dsharp -a {0} {0} -l {1} -print kabs".format(a * 1.0e4, Lambda_obs)
        out = subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout
        Kabs.append(float(out.strip().split()[-1]))
    return np.array(Kabs)


def calc_beam():
    # ガウスビームの等間隔格子 r_u と行列 W[i, j] : r_u[j] の環が r_u[i] に寄与する重み (i0e で桁あふれを防ぐ)
    s_b = beam_fwhm_arcsec * distance_pc / (2.0 * np.sqrt(2.0 * np.log(2.0))) * AU   # 1" at d pc = d au
    r_u = np.arange(0.5 * dr_conv, r_conv_max, dr_conv) * AU
    x = r_u[:, None] * r_u[None, :] / s_b**2
    W = (r_u[None, :] / s_b**2) * np.exp(-(r_u[:, None] - r_u[None, :])**2 / (2.0 * s_b**2)) * i0e(x) * dr_conv * AU
    return r_u, W


def Planck_Func_nu(T, lambda_obs_micron):
    h, c, k = 6.62607015e-27, 2.99792458e10, 1.380649e-16
    L = lambda_obs_micron * 1.0e-4
    expo = np.clip(h * c / (k * T * L), None, 700.0)
    return 2.0 * h * c / L**3 / (np.exp(expo) - 1.0)


def find_radii(r, I, sigma_ratio, Hg=None):
    # Hg を渡さない場合 (cell 47 など) は use_width_cut をかけない
    # r_disk : 成長曲線法
    F = np.concatenate(([0.0], np.cumsum(0.5 * (I[1:] * r[1:] + I[:-1] * r[:-1]) * np.diff(r)))) * 2.0 * np.pi
    r_disk = np.interp(flux_frac * F[-1], F, r)

    # D(r) = dlnI/dr の符号変化で極大(リング)・極小(ギャップ)
    D = np.gradient(np.log(I), r)
    k = np.where(np.sign(D[:-1]) != np.sign(D[1:]))[0]
    i_ext = [i if I[i] * (1 if D[i] > 0 else -1) >= I[i + 1] * (1 if D[i] > 0 else -1) else i + 1 for i in k]
    kind = ["ring" if D[i] > 0 else "gap" for i in k]

    rings, gaps, edges_in, edges_out, depths = [], [], [], [], []
    for n, (i, kd) in enumerate(zip(i_ext, kind)):
        if kd != "gap" or n + 1 >= len(kind) or kind[n + 1] != "ring":
            continue
        i_ring = i_ext[n + 1]
        if use_rdisk_cut and r[i_ring] > r_disk:
            continue
        if use_delta_cut and I[i_ring] / I[i] < delta_min:
            continue
        if use_sigma_cut and sigma_ratio[i] >= sigma_ratio_max:
            continue
        if use_width_cut and Hg is not None and r[i_ring] - r[i] < w_min_H * Hg[i]:
            continue
        gaps.append(r[i])
        rings.append(r[i_ring])
        depths.append(I[i_ring] / I[i])

        # ギャップの縁
        i_peak = i_ext[n - 1] if n >= 1 and kind[n - 1] == "ring" else 0
        I_edge = 0.5 * (I[i_ring] + I[i])                 # Case 1
        if I[i_peak] < I_edge:
            I_edge = 0.5 * (I[i_peak] + I[i])             # Case 2
        edges_in.append(crossing(r, I, I_edge, i_peak, i, first=True))
        edges_out.append(crossing(r, I, I_edge, i, i_ring, first=False))
    return r_disk, rings, gaps, edges_in, edges_out, depths


def crossing(r, I, I_edge, i0, i1, first):
    # r[i0]..r[i1] で I = I_edge を横切る位置 (線形補間). first=True: 最小の r, False: 最大の r
    j = np.where((I[i0:i1] - I_edge) * (I[i0 + 1:i1 + 1] - I_edge) <= 0.0)[0] + i0
    if len(j) == 0:
        return np.nan
    j = j[0] if first else j[-1]
    if I[j + 1] == I[j]:
        return r[j]
    return r[j] + (I_edge - I[j]) / (I[j + 1] - I[j]) * (r[j + 1] - r[j])


# ===== ギャップの判定: Sigma (定義は analysis_obs_theory.ipynb の旧 cell 71) =====
def find_extrema(r, S):
    # 極値 (リング: 極大, ギャップ: 極小) のセル番号と種類を r の順に返す
    lnS = np.log(np.maximum(S, 1.0e-300))
    nw = int(np.ceil(D_SIGMA))
    if use_prominence:
        # 平滑化した ln Sigma に find_peaks (prominence >= prom_min) をかける
        lnS_s = gaussian_filter1d(lnS, sigma=D_SIGMA, mode="nearest")
        k_ring, _ = find_peaks(lnS_s, prominence=prom_min)
        k_gap, _ = find_peaks(-lnS_s, prominence=prom_min)
        k = np.concatenate((k_ring, k_gap))
        kind = np.array(["ring"] * len(k_ring) + ["gap"] * len(k_gap))
        order = np.argsort(k)
        k, kind = k[order], list(kind[order])
    else:
        # 平滑化した D = d ln Sigma/dr の符号変化
        D = gaussian_filter1d(np.gradient(lnS, r), sigma=D_SIGMA, mode="nearest")
        k = np.where(np.sign(D[:-1]) != np.sign(D[1:]))[0]
        kind = ["ring" if D[i] > 0 else "gap" for i in k]
    # 極値の位置: 候補の前後 +-D_SIGMA セルで生の Sigma が最大 (ring) / 最小 (gap) のセル
    i_ext = []
    for i, kd in zip(k, kind):
        j0, j1 = max(i - nw, 0), min(i + 1 + nw, len(S) - 1)
        seg = S[j0:j1 + 1]
        i_ext.append(j0 + int(np.argmax(seg) if kd == "ring" else np.argmin(seg)))
    return i_ext, kind


def find_ring_gap(r, S, Hg):
    # Sigma(r) のリング・ギャップと縁 (cell 45 の find_radii と同じ縁の決め方. r_disk の条件はなし)
    # 各ギャップは外側で最も近いリングと組にし, r_peak は内側で最も近いリング (なければ範囲の内端)
    i_ext, kind = find_extrema(r, S)
    rings, gaps, edges_in, edges_out = [], [], [], []
    for n, (i, kd) in enumerate(zip(i_ext, kind)):
        if kd != "gap":
            continue
        outer = [i_ext[m] for m in range(n + 1, len(kind)) if kind[m] == "ring"]
        if len(outer) == 0:
            continue
        i_ring = outer[0]
        if S[i] <= 0.0 or S[i_ring] / S[i] < sig_delta_min:
            continue
        if r[i_ring] - r[i] < sig_w_min_H * Hg[i]:
            continue
        inner = [i_ext[m] for m in range(n) if kind[m] == "ring"]
        i_peak = inner[-1] if len(inner) > 0 else 0
        S_edge = 0.5 * (S[i_ring] + S[i])                 # Case 1
        if S[i_peak] < S_edge:
            S_edge = 0.5 * (S[i_peak] + S[i])             # Case 2
        gaps.append(r[i])
        rings.append(r[i_ring])
        edges_in.append(crossing(r, S, S_edge, i_peak, i, first=True))
        edges_out.append(crossing(r, S, S_edge, i, i_ring, first=False))
    return rings, gaps, edges_in, edges_out


# ===== ギャップの判定: F_net = 0 =====
def calc_F_net(r, sig, vgr, B, eta, B_round=1.0e-2):
    # 内部面 r_bod[1..n-1] での F_net = F_adv + F_diff (cell 58 と同じ再構成, なましなし, 丸め誤差以下の F_diff = 0)
    # B_round : F_diff の丸め誤差の見積もりに使う B の相対誤差
    q = r[1] / r[0]
    r_bod = r[1:] / np.sqrt(q)
    w = np.sqrt(sig[1:]) / (np.sqrt(sig[1:]) + np.sqrt(sig[:-1]))
    v_bod = w * vgr[1:] + (1.0 - w) * vgr[:-1]
    B_bod = w * B[1:] + (1.0 - w) * B[:-1]
    eta_bod = w * eta[1:] + (1.0 - w) * eta[:-1]
    s = r_bod - r[:-1]
    t = r[1:] - r_bod
    dBdr = (s*s * B[1:] + (t*t - s*s) * B_bod - t*t * B[:-1]) / (s * t * (s + t))
    dBdr = np.where(B[1:] == B[:-1], 0.0, dBdr)
    dB = B_round * np.abs(B)
    err_F = eta_bod * (s*s * dB[1:] + np.abs(t*t - s*s) * (w * dB[1:] + (1.0 - w) * dB[:-1])
                       + t*t * dB[:-1]) / (s * t * (s + t))
    F_adv = -v_bod * B_bod
    F_diff = eta_bod * dBdr
    err_F = err_F
    F_diff = np.where(np.abs(F_diff) > err_F, F_diff, 0.0)
    return r_bod, F_adv + F_diff


def find_wind_edges(r, T, sig, vgr, B, eta):
    # r_in = スノーライン, r_out = r_in より外側 (r <= r_search_max) で F_net が符号を変える最も外側の半径 (線形補間) [cm]. なければ NaN
    r_in = find_snowline(r, T) * AU
    r_bod, F_net = calc_F_net(r, sig, vgr, B, eta)
    k = np.where((F_net[:-1] * F_net[1:] < 0.0) & (r_bod[:-1] > r_in) & (r_bod[1:] <= r_search_max * AU))[0]
    if np.isnan(r_in) or len(k) == 0:
        return r_in, np.nan
    k = k[-1]
    return r_in, r_bod[k] + F_net[k] / (F_net[k] - F_net[k + 1]) * (r_bod[k + 1] - r_bod[k])